<a href="https://colab.research.google.com/github/gunnsmart/skills/blob/main/AI_Image_Upscaler_Easy_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖼️ AI Image Upscaler — ขยายภาพคมชัดด้วย AI (ฟรี)

โปรแกรมนี้ใช้ **Real-ESRGAN** ขยายภาพให้ใหญ่และคมชัดขึ้น 2-4 เท่า ใช้งานได้ฟรีบน Google Colab

### 📋 วิธีใช้ (ทำตาม 3 ขั้นตอน)

**ขั้นตอนที่ 1:** ตั้งค่า GPU ให้ฟรีและเร็วขึ้น
- ไปที่เมนู `Runtime` → `Change runtime type` → เลือก `T4 GPU` → กด `Save`

**ขั้นตอนที่ 2:** กดปุ่ม ▶️ ที่เซลล์แรกด้านล่าง เพื่อติดตั้งระบบ (รอประมาณ 1 นาที ทำครั้งเดียวต่อ session)

**ขั้นตอนที่ 3:** กดปุ่ม ▶️ ที่เซลล์ที่สอง เลือกขนาดที่ต้องการ แล้วอัปโหลดรูป (เลือกได้หลายรูปพร้อมกัน) — เสร็จแล้วไฟล์ ZIP จะดาวน์โหลดให้อัตโนมัติ

> 💡 ทำรูปชุดใหม่? แค่รันเซลล์ที่ 2 ซ้ำได้เลย ไม่ต้องรันเซลล์แรกใหม่

---


In [ ]:
#@title 📥 ขั้นตอนที่ 1: กดปุ่ม Play ซ้ายมือเพื่อเตรียมระบบ (รันแค่ครั้งเดียวตอนเปิดหน้าเว็บ) { display-mode: "form" }
import subprocess, sys, os, site, pathlib

# ตรวจสอบว่ามี GPU หรือไม่ เพื่อเตือนผู้ใช้ล่วงหน้า
gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print('⚠️  ยังไม่ได้เปิดใช้ GPU! ระบบจะทำงานได้ แต่จะช้ากว่าปกติมาก')
    print('   วิธีแก้: เมนู Runtime > Change runtime type > เลือก T4 GPU > Save > แล้วกด Play ใหม่อีกครั้ง')
    print()
else:
    print('✅ ตรวจพบ GPU พร้อมใช้งาน ระบบจะทำงานได้เร็ว')
    print()

print('⏳ กำลังเตรียมระบบ AI กรุณารอสักครู่ (ขั้นตอนนี้ใช้เวลาประมาณ 1 นาที)...')
pkgs = ['basicsr', 'facexlib', 'gfpgan', 'realesrgan', 'Pillow', 'opencv-python']
install = subprocess.run([sys.executable, '-m', 'pip', 'install'] + pkgs + ['-q'], capture_output=True, text=True)

for sp in site.getsitepackages():
    deg = pathlib.Path(sp) / 'basicsr/data/degradations.py'
    if deg.exists():
        txt = deg.read_text()
        if 'functional_tensor' in txt:
            deg.write_text(txt.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            ))
        break

if not os.path.exists('/content/Real-ESRGAN'):
    subprocess.run(['git', 'clone', 'https://github.com/xinntao/Real-ESRGAN.git', '-q'])

os.chdir('/content/Real-ESRGAN')
setup_result = subprocess.run([sys.executable, 'setup.py', 'develop', '-q'], capture_output=True, text=True)

if setup_result.returncode == 0:
    print()
    print('✅ ระบบพร้อมใช้งานแล้ว! เลื่อนลงไปกดปุ่ม Play ที่ขั้นตอนที่ 2 ด้านล่างได้เลย')
else:
    print()
    print('❌ เกิดข้อผิดพลาดระหว่างติดตั้ง ลองกด Runtime > Restart session แล้วรันเซลล์นี้ใหม่อีกครั้ง')
    print(setup_result.stderr[-1000:])

✅ ตรวจพบ GPU พร้อมใช้งาน ระบบจะทำงานได้เร็ว

⏳ กำลังเตรียมระบบ AI กรุณารอสักครู่ (ขั้นตอนนี้ใช้เวลาประมาณ 1 นาที)...

✅ ระบบพร้อมใช้งานแล้ว! เลื่อนลงไปกดปุ่ม Play ที่ขั้นตอนที่ 2 ด้านล่างได้เลย


In [ ]:
#@title 🛠️ ขั้นตอนที่ 2: เลือกตั้งค่า แล้วกด Play เพื่ออัปโหลดรูปภาพ { display-mode: "form" }
import os, subprocess, time, shutil, zipfile
from PIL import Image
from google.colab import files

#@markdown ---
#@markdown ### 🎛️ 1. ต้องการให้รูปขยายใหญ่ขึ้นกี่เท่า:
ขยายขนาดรูป = "4 เท่า (แนะนำสำหรับภาพชัดเจน)" #@param ["2 เท่า (ขยายแบบพอดีๆ)", "4 เท่า (แนะนำสำหรับภาพชัดเจน)"]

#@markdown ### 🤖 2. รูปภาพเป็นประเภทไหน:
ประเภทของรูปภาพ = "ภาพถ่ายทิวทัศน์โฟโต้โทนชัดเจน (ภาพคน / ภาพถ่ายจริง)" #@param ["ภาพถ่ายทิวทัศน์โฟโต้โทนชัดเจน (ภาพคน / ภาพถ่ายจริง)", "ภาพวาดลายเส้นภาพการ์ตูนอนิเมะ 2D (ภาพกราฟิก / โลโก้ / ภาพวาด)"]

#@markdown ### 📂 3. นามสกุลไฟล์ที่ต้องการตอนจบ:
นามสกุลไฟล์ที่ต้องการ = "JPEG" #@param ["JPEG", "PNG"]

#@markdown ---

# แปลงค่าภาษาไทยเป็นระบบหลังบ้าน
SCALE = 4 if "4 เท่า" in ขยายขนาดรูป else 2
ESRGAN_MODEL = 'RealESRGAN_x4plus_anime_6B' if "อนิเมะ" in ประเภทของรูปภาพ else 'RealESRGAN_x4plus'
ext = '.jpg' if นามสกุลไฟล์ที่ต้องการ == 'JPEG' else '.png'

INPUT_FOLDER = '/content/inputs'
OUTPUT_FOLDER = '/content/outputs'

if not os.path.exists('/content/Real-ESRGAN'):
    print('❌ ยังไม่ได้เตรียมระบบ กรุณาเลื่อนขึ้นไปกดปุ่ม Play ที่ขั้นตอนที่ 1 ก่อน')
else:
    for folder in [INPUT_FOLDER, OUTPUT_FOLDER]:
        if os.path.exists(folder):
            shutil.rmtree(folder)
        os.makedirs(folder, exist_ok=True)

    print("📤 จะมีหน้าต่างเด้งขึ้นมา ให้เลือกรูปภาพจากคอมพิวเตอร์ (เลือกพร้อมกันได้หลายรูป):")
    uploaded = files.upload()

    if not uploaded:
        print("❌ ยังไม่ได้เลือกรูปภาพใดๆ กรุณากด Play ใหม่อีกครั้งเพื่อเริ่มงาน")
    else:
        for filename in uploaded.keys():
            current_path = os.path.join('/content/Real-ESRGAN', filename)
            if not os.path.exists(current_path):
                current_path = os.path.join('/content', filename)
            shutil.move(current_path, os.path.join(INPUT_FOLDER, filename))

        imgs = sorted([f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))])
        print(f"\n📥 ดึงรูปเข้าสู่ระบบสำเร็จ {len(imgs)} ภาพ เริ่มส่งให้ AI ขยายขนาดทันที...")

        os.chdir('/content/Real-ESRGAN')
        success_count = 0
        failed = []
        start_time = time.time()

        for idx, src_name in enumerate(imgs):
            src_path = os.path.join(INPUT_FOLDER, src_name)
            src_stem = os.path.splitext(src_name)[0]
            out_name = f"{src_stem}{ext}"
            out_path = os.path.join(OUTPUT_FOLDER, out_name)

            try:
                with Image.open(src_path) as im:
                    w, h = im.size
            except Exception:
                print(f"   ⚠️  ข้ามไฟล์ที่เปิดไม่ได้: {src_name}")
                failed.append(src_name)
                continue

            print(f"🖼️  กำลังทำรูปที่ [{idx+1}/{len(imgs)}]: {src_name}")

            tile_size = 512 if w * h > 3000000 else 0

            cmd = [
                'python', 'inference_realesrgan.py',
                '-n', ESRGAN_MODEL,
                '-i', src_path,
                '-o', '/content/Real-ESRGAN/results',
                '-s', str(SCALE)
            ]
            if tile_size > 0:
                cmd += ['--tile', str(tile_size)]

            os.makedirs('/content/Real-ESRGAN/results', exist_ok=True)
            for f in os.listdir('/content/Real-ESRGAN/results'):
                os.remove(os.path.join('/content/Real-ESRGAN/results', f))

            result = subprocess.run(cmd, capture_output=True, text=True)

            res_files = os.listdir('/content/Real-ESRGAN/results')
            if res_files:
                res_file = os.path.join('/content/Real-ESRGAN/results', res_files[0])
                with Image.open(res_file) as im_out:
                    if นามสกุลไฟล์ที่ต้องการ == 'JPEG':
                        im_out.convert('RGB').save(out_path, 'JPEG', quality=95, dpi=(300, 300), optimize=True)
                    else:
                        im_out.save(out_path, 'PNG', dpi=(300, 300))
                print(f"   ✅ รูปที่ {idx+1} เสร็จเรียบร้อย")
                success_count += 1
            else:
                print(f"   ❌ รูปที่ {idx+1} ทำไม่สำเร็จ: {src_name}")
                failed.append(src_name)

        elapsed = time.time() - start_time
        print(f"\n⏱️  ใช้เวลาทั้งหมด {elapsed:.0f} วินาที | สำเร็จ {success_count}/{len(imgs)} ภาพ")
        if failed:
            print(f"⚠️  ไฟล์ที่ทำไม่สำเร็จ: {', '.join(failed)}")

        if success_count > 0:
            zip_name = f"upscaled_{SCALE}x.zip"
            zip_path = os.path.join('/content', zip_name)
            if os.path.exists(zip_path):
                os.remove(zip_path)

            print(f"\n📦 กำลังมัดรวมรูปภาพทั้งหมดใส่ไฟล์ ZIP...")
            with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for root, dirs, files_in_dir in os.walk(OUTPUT_FOLDER):
                    for file in files_in_dir:
                        zipf.write(os.path.join(root, file), file)

            print(f"📥 ระบบกำลังส่งไฟล์ {zip_name} เซฟลงคอมพิวเตอร์ให้ทันที")
            files.download(zip_path)
        else:
            print("\n❌ เกิดข้อผิดพลาดในการแปลงรูปภาพทุกไฟล์ ลองรันเซลล์ขั้นตอนที่ 1 ใหม่อีกครั้ง แล้วค่อยลองใหม่")